# Deliverable 4 — Paso 4: Evaluación Multi-Agente (`Redactor` QLoRA vs Sistema Completo D3)

Evalúa el sistema de 4 agentes de `Deliverable 3` (`Enrutador` → `Recuperador` ReAct/E5 → `Redactor` → `Verificador`) sobre las **75 preguntas oficiales** (`50` congeladas + `25` difíciles), acoplando el adaptador LoRA público **`Pablox0333/qwen3-4b-redactor-udec-d4` únicamente en el agente `Redactor`**.

- Los agentes `Enrutador`, `Recuperador` y `Verificador` ejecutan dentro de `model.disable_adapter()` usando exactamente los pesos base de `Qwen/Qwen3-4B` (NF4).
- Se mantienen idénticos corpus (`197` unidades / `201` fragmentos), recuperador denso `intfloat/multilingual-e5-small`, prompts v6 (`Deliverable 4/prompts/`), semillas y parámetros de decodificación (`temperature=0.7, top_p=0.8, top_k=20`, `enable_thinking=False`).
- Guardado incremental en Google Drive tras **cada pregunta** (reanudable con `REANUDAR_CARPETA`).

## Paso 0 — Revisar GPU y entorno
En Colab selecciona `Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4 / L4 / A100)`.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    nombre = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {nombre} (sm_{cap[0]}{cap[1]}) | VRAM: {vram} GB")
else:
    raise RuntimeError("No hay GPU activa. Cambia el entorno a GPU en Colab antes de continuar.")

## Paso 1 — Instalar dependencias
Se fija `huggingface-hub>=0.30,<2.0` para compatibilidad con paquetes preinstalados de Colab sin tocar `torch` ni `pandas`.

In [ ]:
%pip install -q -U "transformers>=4.51" peft bitsandbytes accelerate "huggingface-hub>=0.30,<2.0" sentence-transformers
print("Dependencias listas.")

## Paso 2 — Cargar el proyecto desde GitHub y verificar archivos

In [ ]:
import hashlib
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA_GIT = "deliverable-4"
WORKDIR = Path("/content/d4_eval_work")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", RAMA_GIT, REPO_URL, str(WORKDIR)],
    check=True,
)

D4_DIR = WORKDIR / "Deliverable 4"
D3_AGENTES = WORKDIR / "Deliverable 3" / "Deliverable3" / "agentes"
SCRIPT_EVAL = D4_DIR / "scripts" / "evaluar_d4.py"
HF_REPO_ID = "Pablox0333/qwen3-4b-redactor-udec-d4"

for p in (SCRIPT_EVAL, D3_AGENTES / "corpus.json", D3_AGENTES / "test_set_50.csv", D3_AGENTES / "test_set_dificil_25.csv"):
    assert p.is_file(), f"Falta archivo requerido: {p}"

print("Commit clonado:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
print("HF_REPO_ID:", HF_REPO_ID)

## Paso 3 — Configurar respaldo en Drive y smoke check de aislamiento LoRA
Monta Google Drive, crea (o reanuda) la carpeta de corrida y verifica en 1 pregunta (`congelado_50:13`, guardada aparte en `_smoke`) que el adaptador público descarga de Hugging Face, produce logits distintos al base y solo se activa durante llamadas del `Redactor`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
from google.colab import drive

drive.mount("/content/drive")

REANUDAR_CARPETA = None  # <-- Para retomar una corrida interrumpida, pega su ruta completa en Drive
if REANUDAR_CARPETA:
    OUT_DIR = Path(REANUDAR_CARPETA)
else:
    STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    OUT_DIR = Path(f"/content/drive/MyDrive/D4_eval_runs/run_{STAMP}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de salida (Drive):", OUT_DIR)

SMOKE_DIR = OUT_DIR / "_smoke"
cmd_smoke = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(SMOKE_DIR),
    "--qids", "congelado_50:13",
]
print("Ejecutando smoke test:", " ".join(cmd_smoke))
subprocess.run(cmd_smoke, check=True)

tel_smoke = json.loads((SMOKE_DIR / "telemetria_llamadas.json").read_text(encoding="utf-8"))["congelado_50:13"]
print("Llamadas observadas en smoke test:")
for c in tel_smoke:
    print(" ", c["agente"], "| usa_adaptador=", c["usa_adaptador"], "| in=", c["input_tokens"], "| out=", c["output_tokens"], "| s=", c["segundos"])

## Paso 4 — Evaluar las 75 preguntas (`congelado_50` + `dificil_25`)
Guarda `resultados_agentes_D4.csv`, `trazas_D4.json` y `telemetria_llamadas.json` tras cada pregunta. Si la sesión se desconecta, basta volver a ejecutar el Paso 3 con `REANUDAR_CARPETA = str(OUT_DIR)` y re-ejecutar esta celda.

In [ ]:
import subprocess

cmd_eval = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
]
print("Ejecutando evaluación completa (75 preguntas):", " ".join(cmd_eval))
subprocess.run(cmd_eval, check=True)

## Paso 5 — Resumen operacional e inspección de casos históricos D3 v6
La columna `correcto` queda vacía para evaluación humana rigurosa con la misma pauta externa de D2/D3. Aquí resumimos comportamiento operativo y revisamos las 9 preguntas donde D3 v6 falló.

In [ ]:
import pandas as pd

df = pd.read_csv(OUT_DIR / "resultados_agentes_D4.csv", dtype={"id": str}, keep_default_na=False)
print("Total filas:", len(df))
print(df.groupby("set").agg(
    preguntas=("id", "count"),
    ciclos_prom=("ciclos", "mean"),
    con_correccion=("ciclos", lambda c: int((c > 1).sum())),
    fallbacks=("fallbacks", "sum"),
    seg_prom=("segundos", "mean"),
    llamadas_redactor_tot=("llamadas_redactor", "sum"),
    llamadas_base_tot=("llamadas_base", "sum"),
).round(2))

errores_d3 = {
    ("congelado_50", "13"), ("congelado_50", "24"), ("congelado_50", "26"), ("congelado_50", "36"),
    ("dificil_25", "H03"), ("dificil_25", "H07"), ("dificil_25", "H14"), ("dificil_25", "H18"), ("dificil_25", "H25"),
}
print("\n=== Respuestas en los 9 casos que fallaron en D3 v6 ===")
for _, r in df.iterrows():
    if (r["set"], str(r["id"])) in errores_d3:
        print("-" * 72)
        print(f"{r['set']}:{r['id']} | gold={r['gold_dato']} [{r['gold_fuente']}] | aprob={r['aprobado_verificador']} | ciclos={r['ciclos']}")
        print(r["respuesta_sistema"])

## Paso 6 — Comparativa controlada Base vs Fine-tuned (mismo entorno) para análisis y video
Ejecuta los 4 agentes con pesos base vs con LoRA solo en el `Redactor` sobre una selección representativa (`congelado_50:13,congelado_50:24,congelado_50:36,dificil_25:H03,dificil_25:H07,dificil_25:H14`), guardando ambas trazas completas.

In [ ]:
import subprocess

COMPARAR_IDS = "congelado_50:13,congelado_50:24,congelado_50:36,dificil_25:H03,dificil_25:H07,dificil_25:H14"
cmd_comp = [
    "python3",
    str(SCRIPT_EVAL),
    "--hf-repo", HF_REPO_ID,
    "--output-dir", str(OUT_DIR),
    "--solo-comparativa",
    "--comparar-ids", COMPARAR_IDS,
]
print("Ejecutando comparativa lado a lado:", " ".join(cmd_comp))
subprocess.run(cmd_comp, check=True)

## Paso 7 — Empaquetar y descargar resultados de Paso 4

In [ ]:
import zipfile
from google.colab import files

bundle_zip = OUT_DIR / "D4_resultados_eval.zip"
archivos = [
    OUT_DIR / "resultados_agentes_D4.csv",
    OUT_DIR / "trazas_D4.json",
    OUT_DIR / "telemetria_llamadas.json",
    OUT_DIR / "configuracion_evaluacion.json",
    OUT_DIR / "comparativa_base_vs_finetuned.json",
]
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in archivos:
        if f.is_file():
            zf.write(f, arcname=f.name)

print("Paquete generado:", bundle_zip)
files.download(str(bundle_zip))